# 05 - Inference Tren Video Don Le
Notebook chay deepfake detection tren video, gom visualization va batch inference nhieu video.

## Cell 1 - Setup

In [ ]:
# Muc dich: load model tu checkpoint va khai bao VIDEO_PATH.
%pip install mediapipe opencv-python-headless pyyaml pandas matplotlib seaborn tqdm tabulate

import sys
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import yaml
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

sns.set_theme(style='whitegrid', context='notebook')

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'configs').exists() and (PROJECT_ROOT.parent / 'configs').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from preprocess.face_detector import FaceDetector
from preprocess.video_utils import get_video_info, sample_frame_indices, read_frames_at_indices, list_video_files
from models.deepfake_model import DeepfakeDetector

MODEL_CFG_PATH = PROJECT_ROOT / 'configs' / 'model_config.yaml'
TRAIN_CFG_PATH = PROJECT_ROOT / 'configs' / 'train_config.yaml'

with open(MODEL_CFG_PATH, 'r', encoding='utf-8') as f:
    model_cfg = yaml.safe_load(f)
with open(TRAIN_CFG_PATH, 'r', encoding='utf-8') as f:
    train_cfg = yaml.safe_load(f)

# Doi VIDEO_PATH theo file ban muon test.
VIDEO_PATH = Path(r'path/to/your/video.mp4')

ckpt_cfg = train_cfg.get('checkpoint', {})
save_dir = Path(ckpt_cfg.get('save_dir') or (PROJECT_ROOT / 'checkpoints'))
all_ckpts = sorted(save_dir.rglob('*.pth')) if save_dir.exists() else []
if len(all_ckpts) == 0:
    raise RuntimeError(f'Khong tim thay checkpoint trong {save_dir}')

def _ckpt_val_auc(p):
    try:
        obj = torch.load(str(p), map_location='cpu')
        if isinstance(obj, dict):
            return float(obj.get('val_auc', -1e9))
    except Exception:
        pass
    return -1e9

best_ckpt = sorted(all_ckpts, key=_ckpt_val_auc, reverse=True)[0]
CHECKPOINT_PATH = best_ckpt

model_main_cfg = model_cfg.get('model', {})
input_cfg = model_main_cfg.get('input', {})
tf_cfg = model_main_cfg.get('transformer', {})
eff_cfg = model_main_cfg.get('efficientnet', {})

num_frames = int(input_cfg.get('num_frames', model_main_cfg.get('num_frames', 10)))
img_size = int(input_cfg.get('img_size', model_main_cfg.get('image_size', 256)))
model_name = str(eff_cfg.get('model_name', 'efficientnet_b4'))
feat_dim_map = {'efficientnet_b0': 1280, 'efficientnet_b4': 1792}
feat_dim = int(model_main_cfg.get('feat_dim', feat_dim_map.get(model_name, 1792)))
dropout = float(eff_cfg.get('dropout', tf_cfg.get('dropout', 0.3)))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DeepfakeDetector(
    feat_dim=feat_dim,
    d_model=int(tf_cfg.get('d_model', 512)),
    nhead=int(tf_cfg.get('nhead', 8)),
    num_layers=int(tf_cfg.get('num_layers', 4)),
    dropout=dropout,
    num_frames=num_frames,
)
meta = model.load_checkpoint(str(CHECKPOINT_PATH))
model = model.to(device).eval()

print('Device          :', device)
print('Checkpoint      :', CHECKPOINT_PATH)
print('Checkpoint epoch:', meta.get('epoch'))
print('Checkpoint AUC  :', meta.get('val_auc'))
print('VIDEO_PATH      :', VIDEO_PATH)


## Cell 2 - Preprocess Video Input (In-Memory)

In [ ]:
# Muc dich: trich frame tu VIDEO_PATH, detect + align/crop in-memory, khong luu ra disk.
if not VIDEO_PATH.exists():
    raise RuntimeError(f'Khong tim thay video: {VIDEO_PATH}')

video_info = get_video_info(VIDEO_PATH)
print(video_info)

indices = sample_frame_indices(
    total_frames=int(video_info['total_frames']),
    n_samples=num_frames,
    mode='uniform',
)
raw_frames_bgr = read_frames_at_indices(VIDEO_PATH, indices)
if len(raw_frames_bgr) == 0:
    raise RuntimeError('Khong doc duoc frame nao tu video.')

detector_model_path = PROJECT_ROOT / 'preprocess' / 'models' / 'blaze_face_short_range.tflite'
detector = FaceDetector(model_path=detector_model_path, min_detection_confidence=0.6)

cropped_frames_bgr = []
for frame_bgr in raw_frames_bgr:
    det = detector.detect(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
    if det is not None:
        crop = detector.align_and_crop(frame_bgr, det, target_size=(img_size, img_size), crop_margin=0.3)
        if crop is None:
            crop = cv2.resize(frame_bgr, (img_size, img_size), interpolation=cv2.INTER_AREA)
    else:
        crop = cv2.resize(frame_bgr, (img_size, img_size), interpolation=cv2.INTER_AREA)
    cropped_frames_bgr.append(crop)

def bgr_to_norm_tensor(frame_bgr):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    x = torch.from_numpy((rgb.astype(np.float32) / 255.0)).permute(2, 0, 1).contiguous()
    mean = torch.tensor([0.485, 0.456, 0.406], dtype=x.dtype).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], dtype=x.dtype).view(3, 1, 1)
    return (x - mean) / std

frame_tensors = [bgr_to_norm_tensor(f) for f in cropped_frames_bgr]
if len(frame_tensors) < num_frames:
    repeats = int(np.ceil(num_frames / len(frame_tensors)))
    frame_tensors = (frame_tensors * repeats)[:num_frames]
    cropped_frames_bgr = (cropped_frames_bgr * repeats)[:num_frames]
elif len(frame_tensors) > num_frames:
    frame_tensors = frame_tensors[:num_frames]
    cropped_frames_bgr = cropped_frames_bgr[:num_frames]

sequence_tensor = torch.stack(frame_tensors, dim=0)

fig, axes = plt.subplots(1, len(cropped_frames_bgr), figsize=(1.8 * len(cropped_frames_bgr), 2.2))
if len(cropped_frames_bgr) == 1:
    axes = [axes]
for i, ax in enumerate(axes):
    ax.imshow(cv2.cvtColor(cropped_frames_bgr[i], cv2.COLOR_BGR2RGB))
    ax.set_title(f'f{i}', fontsize=8)
    ax.axis('off')
plt.suptitle('Cropped frames (in-memory)')
plt.tight_layout()
plt.show()


## Cell 3 - Inference Tren Video

In [ ]:
# Muc dich: chay model tren sequence frame cua video va in ket qua REAL/FAKE + confidence.
from torch.cuda.amp import autocast

clip_tensor = sequence_tensor.unsqueeze(0).to(device)

t0 = time.perf_counter()
with torch.no_grad():
    with autocast(enabled=device.type == 'cuda'):
        logit = model(clip_tensor).view(-1)[0]
prob_fake = float(torch.sigmoid(logit).item())
infer_time = time.perf_counter() - t0

if prob_fake >= 0.5:
    pred_label = 'FAKE'
    confidence = prob_fake * 100.0
else:
    pred_label = 'REAL'
    confidence = (1.0 - prob_fake) * 100.0

print(f'{pred_label} (confidence: {confidence:.1f}%)')
print(f'Raw fake probability: {prob_fake:.4f}')
print(f'Inference time: {infer_time:.4f}s')


## Cell 4 - Frame-level Analysis

In [ ]:
# Muc dich: danh gia tung frame doc lap bang cach repeat frame do thanh sequence 10 frame.
frame_level_scores = []

with torch.no_grad():
    for frame_tensor in frame_tensors:
        single_clip = frame_tensor.unsqueeze(0).repeat(num_frames, 1, 1, 1).unsqueeze(0).to(device)
        with autocast(enabled=device.type == 'cuda'):
            logit_i = model(single_clip).view(-1)[0]
        prob_i = float(torch.sigmoid(logit_i).item())
        frame_level_scores.append(prob_i)

frame_idx = np.arange(len(frame_level_scores))
plt.figure(figsize=(10, 4.2))
sns.barplot(x=frame_idx, y=frame_level_scores, color='#d55e00')
plt.axhline(0.5, linestyle='--', color='gray', linewidth=1)
plt.title('Frame-level Fake Probability (repeat frame -> sequence)')
plt.xlabel('Frame index')
plt.ylabel('Fake probability')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

most_suspicious_idx = int(np.argmax(frame_level_scores))
print('Frame lo nhat (fake score cao nhat):', most_suspicious_idx)
print('Score:', frame_level_scores[most_suspicious_idx])


## Cell 5 - Visualize Voi Overlay va Luu output.mp4

In [ ]:
# Muc dich: ve bbox + score tren tung frame video goc va luu output.mp4.
OUTPUT_VIDEO = PROJECT_ROOT / 'output.mp4'

cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise RuntimeError(f'Khong mo duoc video: {VIDEO_PATH}')

fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 25.0
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(str(OUTPUT_VIDEO), fourcc, float(fps), (w, h))

def infer_frame_score(frame_bgr):
    x = bgr_to_norm_tensor(cv2.resize(frame_bgr, (img_size, img_size), interpolation=cv2.INTER_AREA))
    clip = x.unsqueeze(0).repeat(num_frames, 1, 1, 1).unsqueeze(0).to(device)
    with torch.no_grad():
        with autocast(enabled=device.type == 'cuda'):
            lg = model(clip).view(-1)[0]
    return float(torch.sigmoid(lg).item())

for _ in tqdm(range(max(total_frames, 0)), desc='Overlay video'):
    ok, frame_bgr = cap.read()
    if not ok or frame_bgr is None:
        break

    det = detector.detect(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
    score = infer_frame_score(frame_bgr)

    pred = 'FAKE' if score >= 0.5 else 'REAL'
    conf = score if score >= 0.5 else (1.0 - score)
    color = (0, 0, 255) if pred == 'FAKE' else (0, 180, 0)

    frame_out = frame_bgr.copy()
    if det is not None:
        bbox = det.bounding_box
        x1 = int(max(0, bbox.origin_x))
        y1 = int(max(0, bbox.origin_y))
        x2 = int(min(w - 1, bbox.origin_x + bbox.width))
        y2 = int(min(h - 1, bbox.origin_y + bbox.height))
        cv2.rectangle(frame_out, (x1, y1), (x2, y2), color, 2)

    txt = f'{pred} | conf={conf*100:.1f}%'
    cv2.putText(frame_out, txt, (15, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2, cv2.LINE_AA)
    writer.write(frame_out)

cap.release()
writer.release()

print('Saved overlay video to:', OUTPUT_VIDEO)


## Cell 6 - Batch Inference Nhieu Video

In [ ]:
# Muc dich: xu ly tat ca video trong folder va xuat CSV ket qua.
VIDEO_FOLDER = Path(r'path/to/your/video_folder')
OUTPUT_CSV = PROJECT_ROOT / 'batch_inference_results.csv'

def infer_single_video(video_path: Path):
    t0 = time.perf_counter()
    info = get_video_info(video_path)
    total_frames = int(info['total_frames'])
    if total_frames <= 0:
        return {'video_path': str(video_path), 'prediction': 'ERROR', 'confidence': 0.0, 'inference_time': 0.0}

    idx = sample_frame_indices(total_frames=total_frames, n_samples=num_frames, mode='uniform')
    frames = read_frames_at_indices(video_path, idx)
    if len(frames) == 0:
        return {'video_path': str(video_path), 'prediction': 'ERROR', 'confidence': 0.0, 'inference_time': 0.0}

    crops = []
    for fr in frames:
        dt = detector.detect(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
        if dt is not None:
            cp = detector.align_and_crop(fr, dt, target_size=(img_size, img_size), crop_margin=0.3)
        else:
            cp = None
        if cp is None:
            cp = cv2.resize(fr, (img_size, img_size), interpolation=cv2.INTER_AREA)
        crops.append(cp)

    tensors = [bgr_to_norm_tensor(c) for c in crops]
    if len(tensors) < num_frames:
        rep = int(np.ceil(num_frames / len(tensors)))
        tensors = (tensors * rep)[:num_frames]
    else:
        tensors = tensors[:num_frames]

    clip = torch.stack(tensors, dim=0).unsqueeze(0).to(device)

    with torch.no_grad():
        with autocast(enabled=device.type == 'cuda'):
            lg = model(clip).view(-1)[0]
    prob_fake = float(torch.sigmoid(lg).item())

    pred = 'FAKE' if prob_fake >= 0.5 else 'REAL'
    conf = prob_fake if pred == 'FAKE' else (1.0 - prob_fake)
    t = time.perf_counter() - t0

    return {
        'video_path': str(video_path),
        'prediction': pred,
        'confidence': conf,
        'inference_time': t,
    }

if not VIDEO_FOLDER.exists():
    raise RuntimeError(f'Khong tim thay folder: {VIDEO_FOLDER}')

video_list = list_video_files(VIDEO_FOLDER)
print('So video can infer:', len(video_list))

batch_rows = []
for vp in tqdm(video_list, desc='Batch inference'):
    batch_rows.append(infer_single_video(vp))

batch_df = pd.DataFrame(batch_rows)
batch_df.to_csv(OUTPUT_CSV, index=False)
display(batch_df.head())
print('Saved CSV:', OUTPUT_CSV)
